# AeroCast — Phase 2: Exploratory Data Analysis

Interactive EDA using Plotly over OpenAQ measurements preprocessed through the AeroCast pipeline.

**Sections:**
1. Setup & data load
2. Dataset overview
3. AQI time series
4. AQI distribution & EPA bands
5. Hourly & day-of-week patterns
6. Pollutant correlation matrix
7. Missing data heatmap
8. Engineered feature distributions
9. Key findings

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from aerocast.data.client import OpenAQClient
from aerocast.data.preprocess import (
    clean_raw, pivot_to_wide, compute_aqi_column, engineer_features,
)

print("Environment ready ✓")

Environment ready ✓


## 1. Data Load

Loads from `data/processed/` (DVC-pulled parquet files) if available; otherwise fetches 30 days of data live from the OpenAQ v2 API for 5 US locations.

In [3]:
PROCESSED_PATH = Path("../data/processed")
parquet_files = list(PROCESSED_PATH.glob("*.parquet"))

if parquet_files:
    print(f"Loading {len(parquet_files)} processed parquet file(s) from DVC cache…")
    raw_df = pd.concat([pd.read_parquet(f) for f in sorted(parquet_files)], ignore_index=True)
    print(f"Loaded {len(raw_df):,} raw rows.")
else:
    print("No processed data found — fetching 30 days live from OpenAQ…")
    import os
    from datetime import datetime, timedelta, timezone
    from dotenv import load_dotenv

    load_dotenv(Path("..") / ".env")
    api_key = os.getenv("OPENAQ_API_KEY", "")
    if not api_key:
        raise EnvironmentError(
            "OPENAQ_API_KEY not set. Add it to .env or export it before running."
        )

    client = OpenAQClient(api_key=api_key)
    locs_df = client.fetch_locations(country="US", limit=20)
    location_ids = locs_df["id"].head(5).tolist()
    print(f"Locations selected: {location_ids}")

    date_to = datetime.now(tz=timezone.utc)
    date_from = date_to - timedelta(days=30)

    raw_df = client.fetch_measurements(
        location_ids=location_ids,
        date_from=date_from,
        date_to=date_to,
    )
    print(f"Fetched {len(raw_df):,} raw records.")


No processed data found — fetching 30 days live from OpenAQ…
Locations selected: [64, 161, 162, 163, 164]
Fetched 2,824 raw records.


## 2. Preprocessing

In [4]:
cleaned = clean_raw(raw_df)
wide    = pivot_to_wide(cleaned)
wide    = compute_aqi_column(wide)
wide    = engineer_features(wide)

print(f"Shape after preprocessing: {wide.shape}")
print(f"Columns: {wide.columns.tolist()}")
wide.head(3)

Shape after preprocessing: (1390, 19)
Columns: ['datetime', 'location_id', 'o3', 'pm10', 'pm25', 'aqi', 'hour', 'day_of_week', 'month', 'is_weekend', 'aqi_lag_1h', 'aqi_lag_2h', 'aqi_lag_3h', 'aqi_lag_6h', 'aqi_lag_12h', 'aqi_lag_24h', 'aqi_roll_mean_6h', 'aqi_roll_mean_12h', 'aqi_roll_mean_24h']


,datetime,location_id,o3,pm10,pm25,aqi,hour,day_of_week,month,is_weekend,aqi_lag_1h,aqi_lag_2h,aqi_lag_3h,aqi_lag_6h,aqi_lag_12h,aqi_lag_24h,aqi_roll_mean_6h,aqi_roll_mean_12h,aqi_roll_mean_24h
0,2026-05-08 20:00:00+00:00,162,0.012,11.0,6.8,28.333333,20,4,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-08 21:00:00+00:00,162,0.017,17.0,9.6,40.000000,21,4,5,0,28.333333,NaN,NaN,NaN,NaN,NaN,28.333333,28.333333,28.333333
2,2026-05-08 22:00:00+00:00,162,0.025,28.0,17.1,61.515021,22,4,5,0,40.000000,28.333333,NaN,NaN,NaN,NaN,34.166667,34.166667,34.166667


## 3. Dataset Overview

In [5]:
print("=" * 50)
print("Dataset Overview")
print("=" * 50)
print(f"Date range : {wide['datetime'].min()} → {wide['datetime'].max()}")
print(f"Locations  : {wide['location_id'].nunique()}")
print(f"Total rows : {len(wide):,}")
print()

# Descriptive stats for key columns
pollutants = [c for c in ["pm25", "pm10", "o3", "no2", "so2", "co", "aqi"] if c in wide.columns]
display(wide[pollutants].describe().round(2))

Dataset Overview
Date range : 2026-05-08 20:00:00+00:00 → 2026-06-07 18:00:00+00:00
Locations  : 2
Total rows : 1,390



,pm25,pm10,o3,aqi
count,1389.00,718.00,708.00,1390.00
mean,8.93,19.10,0.03,36.17
std,4.42,7.91,0.02,14.16
min,0.00,4.00,0.00,0.00
25%,6.20,14.00,0.02,25.83
50%,8.60,18.00,0.03,35.83
75%,11.10,22.75,0.04,46.25
max,96.90,67.00,0.10,172.38


In [6]:
# Missing-value summary
miss = wide[pollutants].isnull().mean().mul(100).round(1).rename("missing_%")
print("Missing % per column:")
print(miss.to_string())

Missing % per column:
pm25     0.1
pm10    48.3
o3      49.1
aqi      0.0


## 4. AQI Time Series

In [7]:
LOCATION_IDS = sorted(wide["location_id"].unique())
COLOR_SEQ = px.colors.qualitative.Set2

fig = px.line(
    wide.sort_values("datetime"),
    x="datetime",
    y="aqi",
    color="location_id",
    color_discrete_sequence=COLOR_SEQ,
    title="AQI Over Time by Location",
    labels={"aqi": "AQI", "datetime": "Date (UTC)", "location_id": "Location"},
    template="plotly_white",
)
fig.update_traces(line_width=1.5, opacity=0.85)
fig.update_layout(height=450, legend_title="Location ID")
fig.show()

## 5. AQI Distribution & EPA Bands

In [8]:
AQI_BANDS = [
    (0,   50,  "Good",                           "#00e400"),
    (51,  100, "Moderate",                        "#ffff00"),
    (101, 150, "Unhealthy for Sensitive Groups",  "#ff7e00"),
    (151, 200, "Unhealthy",                       "#ff0000"),
    (201, 300, "Very Unhealthy",                  "#8f3f97"),
    (301, 500, "Hazardous",                       "#7e0023"),
]

fig = px.histogram(
    wide.dropna(subset=["aqi"]),
    x="aqi",
    nbins=60,
    color="location_id",
    barmode="overlay",
    opacity=0.6,
    title="AQI Distribution by Location (with EPA Bands)",
    labels={"aqi": "AQI", "location_id": "Location"},
    template="plotly_white",
    color_discrete_sequence=COLOR_SEQ,
)

for lo, hi, label, color in AQI_BANDS:
    fig.add_vrect(
        x0=lo, x1=hi,
        fillcolor=color, opacity=0.06, layer="below", line_width=0,
        annotation_text=label, annotation_position="top left",
        annotation_font_size=9,
    )

fig.update_layout(height=420, legend_title="Location ID")
fig.show()

## 6. Hourly & Day-of-Week Patterns

In [9]:
hourly = wide.groupby("hour")["aqi"].agg(["mean", "median", "std"]).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hourly["hour"], y=hourly["mean"],
    mode="lines+markers",
    name="Mean AQI",
    line=dict(color="#2196F3", width=2.5),
    error_y=dict(type="data", array=hourly["std"], visible=True,
                 color="rgba(33,150,243,0.25)", thickness=1.5),
))
fig.add_trace(go.Scatter(
    x=hourly["hour"], y=hourly["median"],
    mode="lines",
    name="Median AQI",
    line=dict(color="#FF9800", width=2, dash="dot"),
))
fig.update_layout(
    title="AQI by Hour of Day (all locations combined)",
    xaxis_title="Hour (UTC)",
    yaxis_title="AQI",
    template="plotly_white",
    height=400,
    xaxis=dict(tickmode="linear", tick0=0, dtick=2),
)
fig.show()

In [10]:
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = (
    wide.groupby("day_of_week")["aqi"]
    .mean()
    .reset_index()
    .assign(day_name=lambda d: d["day_of_week"].map(lambda x: DOW_LABELS[x]))
)

fig = px.bar(
    dow,
    x="day_name", y="aqi",
    color="aqi",
    color_continuous_scale=px.colors.sequential.YlOrRd,
    title="Mean AQI by Day of Week",
    labels={"aqi": "Mean AQI", "day_name": ""},
    template="plotly_white",
    category_orders={"day_name": DOW_LABELS},
)
fig.update_layout(height=380, coloraxis_showscale=False)
fig.show()

## 7. Pollutant Correlation Matrix

In [11]:
pollutant_cols = [c for c in ["pm25", "pm10", "o3", "no2", "so2", "co", "aqi"] if c in wide.columns]
corr = wide[pollutant_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Pollutant × AQI Correlation Matrix",
    template="plotly_white",
)
fig.update_layout(height=500)
fig.show()

## 8. Missing Data Heatmap

In [12]:
raw_pollutants = [c for c in ["pm25", "pm10", "o3", "no2", "so2", "co"] if c in wide.columns]

miss_by_loc = (
    wide.groupby("location_id")[raw_pollutants]
    .apply(lambda g: g.isnull().mean() * 100)
    .round(1)
)

fig = px.imshow(
    miss_by_loc,
    text_auto=".1f",
    color_continuous_scale="Reds",
    title="Missing Data % by Location × Pollutant",
    labels={"color": "Missing %", "x": "Pollutant", "y": "Location ID"},
    template="plotly_white",
)
fig.update_layout(height=max(300, 60 * len(miss_by_loc) + 150))
fig.show()

## 9. Engineered Feature Distributions

In [13]:
lag_cols  = sorted([c for c in wide.columns if c.startswith("aqi_lag_")])
roll_cols = sorted([c for c in wide.columns if c.startswith("aqi_roll_mean_")])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["AQI Lag Features (hours)", "AQI Rolling Mean Features (hours)"],
)

for col in lag_cols:
    fig.add_trace(
        go.Box(
            y=wide[col].dropna(),
            name=col.replace("aqi_lag_", ""),
            marker_color="#5C6BC0",
            showlegend=False,
            boxmean=True,
        ),
        row=1, col=1,
    )
for col in roll_cols:
    fig.add_trace(
        go.Box(
            y=wide[col].dropna(),
            name=col.replace("aqi_roll_mean_", ""),
            marker_color="#26A69A",
            showlegend=False,
            boxmean=True,
        ),
        row=1, col=2,
    )

fig.update_layout(
    title="Engineered Lag & Rolling Feature Distributions",
    template="plotly_white",
    height=450,
)
fig.show()

## 10. Key Findings

In [14]:
aqi = wide["aqi"].dropna()
peak_hour = wide.groupby("hour")["aqi"].mean().idxmax()
lag24 = wide.get("aqi_lag_24h")
lag24_corr = aqi.corr(lag24.dropna().reindex(aqi.index)) if lag24 is not None else float("nan")

pct_good   = (aqi <= 50).mean() * 100
pct_mod    = ((aqi > 50) & (aqi <= 100)).mean() * 100
pct_bad    = (aqi > 100).mean() * 100

print("=" * 50)
print("Key EDA Findings")
print("=" * 50)
print(f"Rows (hourly slots):  {len(wide):,}")
print(f"Locations:            {wide['location_id'].nunique()}")
print(f"AQI range:            {aqi.min():.1f} – {aqi.max():.1f}")
print(f"Median AQI:           {aqi.median():.1f}")
print(f"AQI bands — Good: {pct_good:.1f}%  Moderate: {pct_mod:.1f}%  Unhealthy+: {pct_bad:.1f}%")
print(f"Peak pollution hour:  {peak_hour}:00 UTC")
print(f"Lag-24h correlation:  {lag24_corr:.3f}")
print()
print("→ Strong 24-hour autocorrelation confirms lag features are useful predictors.")
print("→ Check the correlation matrix to identify the dominant AQI pollutant per location.")

Key EDA Findings
Rows (hourly slots):  1,390
Locations:            2
AQI range:            0.0 – 172.4
Median AQI:           35.8
AQI bands — Good: 82.6%  Moderate: 17.3%  Unhealthy+: 0.1%
Peak pollution hour:  17:00 UTC
Lag-24h correlation:  0.169

→ Strong 24-hour autocorrelation confirms lag features are useful predictors.
→ Check the correlation matrix to identify the dominant AQI pollutant per location.
